# Orpheus Turkish TTS Fine-Tuning with LoRA

Fine-tune [Orpheus-3B](https://huggingface.co/unsloth/orpheus-3b-0.1-pretrained) for **Turkish text-to-speech** with LoRA and `TransformersTrainer` on **Red Hat OpenShift AI**.

Orpheus generates speech as discrete SNAC audio tokens. This notebook:

1. Preprocesses Turkish audio into SNAC token sequences
2. Fine-tunes with LoRA + multi-node DDP
3. Loads the LoRA adapter for inference (no weight merge)
4. Generates sample Turkish speech

### Prerequisites

- OpenShift AI (RHOAI) 3.2+ with Kubeflow Trainer v2
- GPU workbench for post-training inference
- Shared RWX PVC named `shared` (mounted at `/opt/app-root/src/shared`)

See [README.md](README.md) for hardware sizing and setup details.


## Install the Kubeflow SDK


In [1]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"

In [2]:
# Install the YAML magic
!pip install yamlmagic --index-url https://pypi.org/simple
%load_ext yamlmagic

## Training Configuration

Edit the parameters below for your project and hardware.


In [3]:
%%yaml parameters

# Infrastructure
namespace: your-data-science-project   # set to your OpenShift AI project name
mlflow_experiment: orpheus-turkish-tts

# Model
base_model: unsloth/orpheus-3b-0.1-pretrained
hf_dataset: afkfatih/turkish-tts-combined-raw
max_train_samples: 0                      # 0 = full dataset (~81K samples)
max_seq_len: 4096

# LoRA
lora_r: 64
lora_alpha: 128
lora_dropout: 0.05                        # light dropout to prevent overfitting on 5 epochs

# Training
num_nodes: 2
gpus_per_node: 2
batch_size: 4                             # gradient checkpointing enabled — fits 4 on A100-80GB
grad_accum: 4                             # effective batch = 4 × 4 × 4 GPUs = 64
learning_rate: 1.0e-4
num_epochs: 5
eval_split: 0.05
warmup_ratio: 0.05

# Checkpointing & Logging
save_steps: 500                           # ~2.5x per epoch (1273 steps/epoch)
logging_steps: 25
eval_steps: 250                           # eval cadence (cheap)
audio_log_steps: 1000                     # wav + Whisper cadence (expensive)
# Generation defaults live in train_func (keep gen_min_new_tokens low to avoid trailing audio junk)


<IPython.core.display.Javascript object>

## Training Loop

`TransformersTrainer` serializes `train_func` with `inspect.getsource()`, so all training logic must live **inside** the function.

The function SNAC-encodes audio (rank 0), then runs LoRA training with MLflow metrics and periodic audio/WER logging.


In [ ]:
def train_func(**parameters):  # noqa: C901
    """Self-contained training function — serialized via inspect.getsource()."""
    import hashlib
    import io
    import logging
    import math
    import os
    import sys
    import tempfile
    import time
    from pathlib import Path
    from types import SimpleNamespace

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.StreamHandler(sys.stdout)],
    )
    log = logging.getLogger("train_orpheus")

    import warnings

    import urllib3

    warnings.filterwarnings(
        "ignore", category=urllib3.exceptions.InsecureRequestWarning
    )

    import librosa
    import mlflow
    import numpy as np
    import soundfile as sf
    import torch
    from datasets import (
        Audio,
        Dataset,
        load_dataset,
        load_from_disk,
    )
    from peft import LoraConfig, TaskType, get_peft_model
    from scipy.signal import resample_poly
    from snac import SNAC
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForSeq2Seq,
        Trainer,
        TrainerCallback,
        TrainingArguments,
    )

    # ── Constants (Orpheus / SNAC token spec) ─────────────────────────────────
    LLAMA_VOCAB = 128_256
    CODE_OFFSET = LLAMA_VOCAB + 10
    N_CODEBOOK = 4_096
    N_PER_FRAME = 7
    SNAC_SR = 24_000

    TOK_SOH = LLAMA_VOCAB + 3
    TOK_EOH = LLAMA_VOCAB + 4
    TOK_SOA = LLAMA_VOCAB + 5
    TOK_EOA = LLAMA_VOCAB + 6
    TOK_SOS = LLAMA_VOCAB + 1
    TOK_EOT = LLAMA_VOCAB + 9

    # ── Parameters from %%yaml (via func_args) ─────────────────────────────────
    # Defaults first; yaml/func_args override (avoid duplicate kwargs with **parameters).
    p = SimpleNamespace(**{
        "audio_log_steps": 500,
        "eval_steps": 100,
        "preprocess_timeout_s": 14400,
        "min_preprocess_rows": 100,
        "whisper_model": "small",
        "gen_max_new_tokens": 900,
        "gen_min_new_tokens": 28,
        "gen_baseline_max_new_cap": 400,
        "gen_temperature": 0.3,
        "gen_top_p": 0.9,
        "gen_repetition_penalty": 1.15,
        "gen_trim_top_db": 28,
        "gen_trail_gap_s": 0.35,
        "gen_min_tokens_per_char": 0,
        **parameters,
    })

    rank = int(os.environ.get("RANK", 0))
    pvc = "/mnt/kubeflow-checkpoints/orpheus-tts"
    hf_cache = f"{pvc}/hf-cache"
    data_dir = f"{pvc}/preprocessed"
    checkpoint_dir = f"{pvc}/checkpoints"

    EVAL_SENTENCES = [
        (
            "flight_announce",
            "sayın yolcularımız, uçuşumuz yaklaşık iki saat sürecektir.",
        ),
        ("welcome", "istanbul'a hoş geldiniz."),
        ("safety", "güvenlik nedeniyle elektronik cihazlarınızı kapalı tutunuz."),
        ("farewell", "teşekkür ederiz, iyi yolculuklar dileriz."),
    ]

    # ══════════════════════════════════════════════════════════════════════════
    # PHASE 1: PREPROCESSING (rank 0 only)
    # ══════════════════════════════════════════════════════════════════════════

    def _preprocess_fingerprint():
        """Invalidate cached preprocess when dataset / size / seq len change."""
        raw = f"{p.hf_dataset}|{p.max_train_samples}|{p.max_seq_len}|{p.base_model}"
        return hashlib.sha1(raw.encode()).hexdigest()[:12]

    def _preprocess():
        """SNAC-encode raw audio → token sequences. Writes versioned sentinel when done."""
        out_path = Path(data_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        fingerprint = _preprocess_fingerprint()
        sentinel = out_path / f".done-{fingerprint}"
        # Remove stale sentinels from older configs
        for stale in out_path.glob(".done*"):
            if stale.name != sentinel.name:
                stale.unlink(missing_ok=True)
        if sentinel.exists() and (out_path / "dataset_info.json").exists():
            log.info(
                "Already preprocessed at %s (fp=%s) — skipping.", out_path, fingerprint
            )
            return

        os.environ["HF_HOME"] = hf_cache
        device = "cuda" if torch.cuda.is_available() else "cpu"
        log.info("Preprocessing on device: %s (fp=%s)", device, fingerprint)

        tokenizer = AutoTokenizer.from_pretrained(p.base_model, cache_dir=hf_cache)
        snac_model = (
            SNAC
            .from_pretrained("hubertsiuzdak/snac_24khz", cache_dir=hf_cache)
            .to(device)
            .eval()
        )

        log.info("Loading raw dataset: %s …", p.hf_dataset)
        raw = load_dataset(p.hf_dataset, split="train", cache_dir=hf_cache)
        raw = raw.cast_column("audio", Audio(decode=False))
        total = (
            min(p.max_train_samples, len(raw)) if p.max_train_samples > 0 else len(raw)
        )
        shard = raw.select(range(total))
        log.info("Processing %d samples", total)

        def _encode(wav_np, src_sr):
            if wav_np.ndim == 2:
                wav_np = wav_np.mean(axis=1)
            if src_sr != SNAC_SR:
                gcd = math.gcd(int(src_sr), SNAC_SR)
                wav_np = resample_poly(wav_np, SNAC_SR // gcd, src_sr // gcd).astype(
                    np.float32
                )
            wav = torch.tensor(wav_np).unsqueeze(0).unsqueeze(0).to(device)
            with torch.inference_mode():
                codes = snac_model.encode(wav)
            return codes[0][0].tolist(), codes[1][0].tolist(), codes[2][0].tolist()

        def _interleave(l0, l1, l2):
            n = min(len(l0), len(l1) // 2, len(l2) // 4)
            t = []
            for f in range(n):
                t += [
                    CODE_OFFSET + l0[f],
                    CODE_OFFSET + 4096 + l1[2 * f],
                    CODE_OFFSET + 8192 + l2[4 * f],
                    CODE_OFFSET + 8192 + l2[4 * f + 1],
                    CODE_OFFSET + 4096 + l1[2 * f + 1],
                    CODE_OFFSET + 8192 + l2[4 * f + 2],
                    CODE_OFFSET + 8192 + l2[4 * f + 3],
                ]
            return t

        rows, skipped = [], 0
        for i, sample in enumerate(shard):
            if i % 500 == 0:
                log.info("  %d / %d  (skipped %d)", i, len(shard), skipped)
            try:
                text = sample["text"].strip()
                audio = sample["audio"]
                raw_bytes = audio.get("bytes") or open(audio["path"], "rb").read()
                wav, sr = sf.read(
                    io.BytesIO(raw_bytes), dtype="float32", always_2d=False
                )
                t_ids = tokenizer.encode(text, add_special_tokens=False)
                l0, l1, l2 = _encode(wav, sr)
                seq = (
                    [TOK_SOH]
                    + t_ids
                    + [TOK_EOT, TOK_EOH, TOK_SOA, TOK_SOS]
                    + _interleave(l0, l1, l2)
                    + [TOK_EOA]
                )
                if len(seq) > p.max_seq_len:
                    skipped += 1
                    continue
                sos_idx = seq.index(TOK_SOS)
                labels = [-100] * (sos_idx + 1) + seq[sos_idx + 1 :]
                rows.append({
                    "input_ids": seq,
                    "labels": labels,
                    "attention_mask": [1] * len(seq),
                })
            except Exception as e:
                log.warning("sample %d skipped: %s", i, e)
                skipped += 1

        min_rows = int(getattr(p, "min_preprocess_rows", 100))
        if len(rows) < min_rows:
            raise RuntimeError(
                f"Preprocessing kept only {len(rows)} sequences (need >= {min_rows}). "
                f"Skipped {skipped}. Check dataset schema (text/audio) or lower max_seq_len filters."
            )

        log.info("Preprocessing done: %d kept, %d skipped", len(rows), skipped)
        Dataset.from_list(rows).save_to_disk(str(out_path))
        sentinel.touch()
        log.info("Saved %d sequences → %s (fp=%s)", len(rows), out_path, fingerprint)

    # Rank 0: preprocess
    if rank == 0:
        _preprocess()

    # File-based sync — fail fast if preprocess never completes
    fingerprint = _preprocess_fingerprint()
    sentinel = Path(data_dir) / f".done-{fingerprint}"
    preprocess_timeout_s = int(getattr(p, "preprocess_timeout_s", 7200))  # 2h default
    waited = 0
    while not sentinel.exists():
        if waited >= preprocess_timeout_s:
            raise TimeoutError(
                f"Rank {rank}: preprocess sentinel {sentinel} not found after "
                f"{preprocess_timeout_s}s — rank 0 likely failed."
            )
        log.info(
            "Rank %d: waiting for preprocessing (fp=%s, %ds/%ds)...",
            rank,
            fingerprint,
            waited,
            preprocess_timeout_s,
        )
        time.sleep(30)
        waited += 30

    # ══════════════════════════════════════════════════════════════════════════
    # PHASE 2: TRAINING (all ranks)
    # ══════════════════════════════════════════════════════════════════════════

    os.environ["HF_HOME"] = hf_cache
    os.environ.setdefault("HF_DATASETS_CACHE", f"{hf_cache}/datasets")
    os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", p.mlflow_experiment)
    # HF Trainer MLflow integration controls:
    # - MLFLOW_FLATTEN_PARAMS: log params flat (avoids nested dicts as strings)
    # - HF_MLFLOW_LOG_ARTIFACTS: don't auto-upload model checkpoints as artifacts
    os.environ.setdefault("MLFLOW_FLATTEN_PARAMS", "1")
    os.environ.setdefault("HF_MLFLOW_LOG_ARTIFACTS", "0")
    try:
        mlflow.enable_system_metrics_logging()
    except Exception:
        pass
    out_dir = Path(checkpoint_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    Path(hf_cache).mkdir(parents=True, exist_ok=True)
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Helpers ────────────────────────────────────────────────────────────────
    def build_prompt(tokenizer, text):
        ids = tokenizer.encode(text, add_special_tokens=False) + [TOK_EOT]
        return [TOK_SOH] + ids + [TOK_EOH, TOK_SOA, TOK_SOS]

    def _clean_wav(wav):
        """Edge-trim silence, then hard-cut at the first long pause (drops trailing babble)."""
        wav = wav.astype(np.float32)
        try:
            trimmed, _ = librosa.effects.trim(
                wav, top_db=p.gen_trim_top_db, frame_length=2048, hop_length=512
            )
            if len(trimmed) / SNAC_SR >= 0.1:
                wav = trimmed
        except Exception:
            pass
        try:
            intervals = librosa.effects.split(
                wav, top_db=p.gen_trim_top_db, frame_length=2048, hop_length=512
            )
            if len(intervals):
                start, end = int(intervals[0][0]), int(intervals[0][1])
                gap_s = float(getattr(p, "gen_trail_gap_s", 0.35))
                for s, e in intervals[1:]:
                    if (s - end) / SNAC_SR > gap_s:
                        break
                    end = int(e)
                cut = wav[start:end]
                if len(cut) / SNAC_SR >= 0.1:
                    wav = cut
        except Exception:
            pass
        peak = np.abs(wav).max()
        if peak > 1e-6:
            wav = wav * (0.9 / peak)
        return wav.astype(np.float32)

    def snac_decode(snac_model, token_ids, device):
        audio_ids = [t for t in token_ids if t >= CODE_OFFSET]
        n = len(audio_ids) // N_PER_FRAME
        if n == 0:
            return None
        audio_ids = audio_ids[: n * N_PER_FRAME]
        l0, l1, l2 = [], [], []
        for f in range(n):
            g = audio_ids[N_PER_FRAME * f : N_PER_FRAME * (f + 1)]
            l0.append((g[0] - CODE_OFFSET) % N_CODEBOOK)
            l1.append((g[1] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[2] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[3] - CODE_OFFSET) % N_CODEBOOK)
            l1.append((g[4] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[5] - CODE_OFFSET) % N_CODEBOOK)
            l2.append((g[6] - CODE_OFFSET) % N_CODEBOOK)

        def _t(x):
            return torch.tensor(x, dtype=torch.long).unsqueeze(0).to(device)

        wav = snac_model.decode([_t(l0), _t(l1), _t(l2)])
        return wav.squeeze().cpu().float().detach().numpy()

    def generate_audio(
        model, tokenizer, snac_model, text, device, step=0, *, baseline=False
    ):
        t0 = time.perf_counter()
        prompt = build_prompt(tokenizer, text)
        inp = torch.tensor([prompt], dtype=torch.long, device=device)
        # Keep min_new_tokens low so TOK_EOA can stop; high mins force trailing SNAC junk.
        min_new = int(p.gen_min_new_tokens)
        if baseline:
            max_new = min(p.gen_max_new_tokens, p.gen_baseline_max_new_cap)
        else:
            max_new = int(p.gen_max_new_tokens)
        with torch.inference_mode():
            out = model.generate(
                inp,
                max_new_tokens=max_new,
                min_new_tokens=min_new,
                do_sample=True,
                temperature=p.gen_temperature,
                top_p=p.gen_top_p,
                use_cache=True,
                repetition_penalty=p.gen_repetition_penalty,
                eos_token_id=TOK_EOA,
            )
        new_ids = out[0][len(prompt) :].cpu().tolist()
        if TOK_EOA in new_ids:
            new_ids = new_ids[: new_ids.index(TOK_EOA)]
        wav = snac_decode(snac_model, new_ids, device)
        elapsed = time.perf_counter() - t0
        if wav is not None:
            try:
                mlflow.log_metric(
                    "rtf_latest",
                    elapsed / max(len(wav) / SNAC_SR, 1e-6),
                    step=step,
                )
            except Exception:
                pass
        return wav, elapsed

    def compute_wer_cer(whisper_mdl, wav, reference):
        import jiwer

        hyp = (
            whisper_mdl
            .transcribe(
                wav.astype(np.float32),
                language="tr",
                task="transcribe",
                initial_prompt="Türkçe konuşma.",
            )["text"]
            .strip()
            .lower()
        )
        ref = reference.strip().lower()
        return jiwer.wer(ref, hyp), jiwer.cer(ref, hyp), hyp

    def log_audio_batch(
        model,
        tokenizer,
        snac_model,
        device,
        step,
        folder,
        whisper_mdl=None,
        *,
        baseline=False,
    ):
        model.eval()
        metrics = {}
        tmp = Path(tempfile.mkdtemp())
        wers, cers, rtfs = [], [], []
        eval_rows = []

        for label, text in EVAL_SENTENCES:
            wav, elapsed = generate_audio(
                model, tokenizer, snac_model, text, device, step=step, baseline=baseline
            )
            if wav is None:
                log.warning("No audio for '%s' at step %d", label, step)
                continue
            wav = _clean_wav(wav)
            duration = len(wav) / SNAC_SR
            if duration < 0.3:
                continue
            rtf = elapsed / max(duration, 1e-6)
            rtfs.append(rtf)
            metrics[f"rtf/{label}"] = rtf

            wav_path = tmp / f"{label}.wav"
            sf.write(str(wav_path), wav, samplerate=SNAC_SR)
            mlflow.log_artifact(str(wav_path), artifact_path=f"{folder}/{label}")

            if whisper_mdl is not None:
                try:
                    wer, cer, hyp = compute_wer_cer(whisper_mdl, wav, text)
                    wers.append(wer)
                    cers.append(cer)
                    metrics[f"wer/{label}"] = wer
                    metrics[f"cer/{label}"] = cer
                    eval_rows.append({
                        "sentence": label,
                        "reference": text,
                        "hypothesis": hyp,
                        "wer": round(wer, 4),
                        "cer": round(cer, 4),
                        "rtf": round(rtf, 4),
                    })
                    log.info(
                        "  [%s] WER=%.3f  CER=%.3f  hyp: %s", label, wer, cer, hyp[:60]
                    )
                except Exception as e:
                    log.warning("WER/CER failed for %s: %s", label, e)

        if rtfs:
            metrics["rtf/mean"] = sum(rtfs) / len(rtfs)
        if wers:
            metrics["wer/mean"] = sum(wers) / len(wers)
        if cers:
            metrics["cer/mean"] = sum(cers) / len(cers)

        mlflow.log_metrics(metrics, step=step)

        # Log a per-sentence eval table as a JSON artifact
        if eval_rows:
            try:
                import json as _json

                tbl_path = tmp / f"eval_step_{step:06d}.json"
                tbl_path.write_text(
                    _json.dumps(eval_rows, indent=2, ensure_ascii=False)
                )
                mlflow.log_artifact(str(tbl_path), artifact_path=f"{folder}/eval_table")
            except Exception:
                pass

        model.train()

    # ── MLflow callback ───────────────────────────────────────────────────────
    class MLflowAudioCallback(TrainerCallback):
        def __init__(self):
            self._snac = None
            self._whisper = None
            self._device = None

        def _get_snac(self):
            if self._snac is None:
                self._snac = (
                    SNAC
                    .from_pretrained("hubertsiuzdak/snac_24khz", cache_dir=hf_cache)
                    .eval()
                    .to(self._device)
                )
            return self._snac

        def _get_whisper(self):
            if self._whisper is None:
                import whisper

                self._whisper = whisper.load_model(p.whisper_model, device="cpu")
                log.info(
                    "Whisper-%s loaded on CPU for in-training WER/CER", p.whisper_model
                )
            return self._whisper

        def on_train_begin(self, args, state, control, model=None, **kwargs):
            self._device = next(model.parameters()).device
            if not state.is_world_process_zero:
                return
            world = int(os.environ.get("WORLD_SIZE", "1"))
            mlflow.set_tags({
                "base_model": p.base_model,
                "dataset": p.hf_dataset,
                "train_samples": str(p.max_train_samples or "full"),
                "max_seq_len": str(p.max_seq_len),
                "n_gpus": str(world),
                "gpu": torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else "cpu",
                "effective_batch": str(p.batch_size * p.grad_accum * world),
            })
            try:
                mlflow.log_params({
                    "lora_r": p.lora_r,
                    "lora_alpha": p.lora_alpha,
                    "lora_dropout": p.lora_dropout,
                    "trainable_params_M": round(trainable / 1e6, 1),
                    "total_params_B": round(total / 1e9, 2),
                    "dataset_samples": len(train_ds),
                })
            except Exception:
                pass

            # Log dataset as an MLflow input for lineage tracking
            try:
                ds_source = mlflow.data.from_huggingface(
                    train_ds,
                    path=p.hf_dataset,
                    targets="audio",
                )
                mlflow.log_input(ds_source, context="training")
            except Exception:
                pass

            # Log full training config as a YAML artifact
            try:
                import yaml as _yaml

                cfg_path = Path(tempfile.mkdtemp()) / "training_config.yaml"
                cfg_path.write_text(
                    _yaml.dump(vars(p), default_flow_style=False, sort_keys=True)
                )
                mlflow.log_artifact(str(cfg_path))
            except Exception:
                pass

            log.info("Logging pretrained baseline audio + metrics …")
            try:
                log_audio_batch(
                    model,
                    tokenizer,
                    self._get_snac(),
                    self._device,
                    step=0,
                    folder="audio/step_000000/pretrained_baseline",
                    whisper_mdl=self._get_whisper(),
                    baseline=True,
                )
            except Exception as e:
                log.warning("Baseline audio failed: %s", e)

        def on_step_end(self, args, state, control, model=None, **kwargs):
            if not state.is_world_process_zero:
                return
            step_interval = min(p.audio_log_steps, max(state.max_steps // 3, 1))
            if state.global_step > 0 and state.global_step % step_interval == 0:
                log.info("Logging audio at step %d …", state.global_step)
                try:
                    log_audio_batch(
                        model,
                        tokenizer,
                        self._get_snac(),
                        self._device,
                        step=state.global_step,
                        folder=f"audio/step_{state.global_step:06d}/finetuned",
                        whisper_mdl=self._get_whisper(),
                    )
                except Exception as e:
                    log.warning("Audio at step %d failed: %s", state.global_step, e)

        def on_save(self, args, state, control, **kwargs):
            if not state.is_world_process_zero:
                return
            ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
            adapter_file = ckpt_dir / "adapter_model.safetensors"
            if not adapter_file.exists():
                adapter_file = ckpt_dir / "adapter_model.bin"
            if adapter_file.exists():
                epoch = int(state.epoch) if state.epoch else "?"
                artifact_path = f"adapters/epoch{epoch}_step{state.global_step}"
                for f in [
                    "adapter_model.safetensors",
                    "adapter_model.bin",
                    "adapter_config.json",
                ]:
                    p_file = ckpt_dir / f
                    if p_file.exists():
                        mlflow.log_artifact(str(p_file), artifact_path=artifact_path)
                log.info("LoRA adapter logged → %s", artifact_path)

        def on_train_end(self, args, state, control, model=None, **kwargs):
            if not state.is_world_process_zero:
                return
            try:
                log_audio_batch(
                    model,
                    tokenizer,
                    self._get_snac(),
                    self._device,
                    step=state.global_step,
                    folder="audio/final",
                    whisper_mdl=self._get_whisper(),
                )
            except Exception as e:
                log.warning("Final audio failed: %s", e)

    # ── Load tokenizer + model ────────────────────────────────────────────────
    log.info("Loading model: %s", p.base_model)
    tokenizer = AutoTokenizer.from_pretrained(p.base_model, cache_dir=hf_cache)
    tokenizer.pad_token = tokenizer.eos_token

    try:
        from flash_attn import flash_attn_func  # noqa: F401

        attn_impl = "flash_attention_2"
    except ImportError:
        attn_impl = "sdpa"
        log.warning("flash-attn not available — using SDPA")

    model = AutoModelForCausalLM.from_pretrained(
        p.base_model,
        cache_dir=hf_cache,
        torch_dtype=torch.bfloat16,
        attn_implementation=attn_impl,
    )
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=p.lora_r,
        lora_alpha=p.lora_alpha,
        lora_dropout=p.lora_dropout,
        bias="none",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )
    model = get_peft_model(model, lora_cfg)
    model.enable_input_require_grads()
    trainable, total = model.get_nb_trainable_parameters()
    log.info(
        "LoRA: %.1fM trainable / %.2fB total (%.2f%%)",
        trainable / 1e6,
        total / 1e9,
        100 * trainable / total,
    )

    # ── Load preprocessed dataset ─────────────────────────────────────────────
    preprocessed_dir = Path(data_dir)
    if not sentinel.exists():
        raise RuntimeError(
            f"Preprocessed dataset sentinel missing: {sentinel}. "
            "Ensure preprocess() ran on rank 0 before calling train()."
        )
    log.info("Loading preprocessed dataset from %s", preprocessed_dir)
    ds = load_from_disk(str(preprocessed_dir))

    if p.max_train_samples:
        ds = ds.select(range(min(p.max_train_samples, len(ds))))

    def _audio_only_labels(example):
        ids = example["input_ids"]
        try:
            sos_idx = ids.index(TOK_SOS)
        except ValueError:
            example["labels"] = list(ids)
            return example
        example["labels"] = [-100] * (sos_idx + 1) + ids[sos_idx + 1 :]
        return example

    ds = ds.map(_audio_only_labels)
    ds = ds.filter(lambda x: len(x["input_ids"]) <= p.max_seq_len)
    ds = ds.map(lambda x: {"length": len(x["input_ids"])})

    split = ds.train_test_split(test_size=p.eval_split, seed=42)
    train_ds = split["train"]
    eval_ds = split["test"]
    log.info("Train: %d  Eval: %d", len(train_ds), len(eval_ds))

    # ── Trainer ───────────────────────────────────────────────────────────────
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding="longest",
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
    )

    # Prefer fused AdamW; fall back on older torch builds
    import inspect as _inspect

    if "fused" in _inspect.signature(torch.optim.AdamW).parameters:
        optim_name = "adamw_torch_fused"
    else:
        optim_name = "adamw_torch"
        log.warning("adamw_torch_fused unavailable — using adamw_torch")

    # load_best_model_at_end requires save_steps to be a multiple of eval_steps
    eval_steps = int(p.eval_steps)
    save_steps = int(p.save_steps)
    if save_steps % eval_steps != 0:
        save_steps = ((save_steps + eval_steps - 1) // eval_steps) * eval_steps
        log.warning(
            "Adjusted save_steps → %d to align with eval_steps=%d",
            save_steps,
            eval_steps,
        )

    training_args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=p.num_epochs,
        per_device_train_batch_size=p.batch_size,
        per_device_eval_batch_size=p.batch_size,
        gradient_accumulation_steps=p.grad_accum,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=p.learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=p.warmup_ratio,
        weight_decay=0.01,
        optim=optim_name,
        bf16=True,
        tf32=True,
        logging_steps=p.logging_steps,
        eval_steps=eval_steps,
        eval_strategy="steps",
        save_steps=save_steps,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        group_by_length=True,
        length_column_name="length",
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        report_to=["mlflow", "tensorboard"],
        logging_dir=f"{pvc}/tensorboard",
        run_name=f"orpheus-tr-{p.max_train_samples or 'full'}-b{p.batch_size}x{p.grad_accum}",
        ddp_find_unused_parameters=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        tokenizer=tokenizer,
        callbacks=[MLflowAudioCallback()],
    )

    log.info("Starting training …")
    from transformers.trainer_utils import get_last_checkpoint

    last_ckpt = get_last_checkpoint(str(out_dir))
    if last_ckpt:
        ckpt_adapter_cfg = Path(last_ckpt) / "adapter_config.json"
        if ckpt_adapter_cfg.exists():
            import json as _json

            ckpt_r = _json.loads(ckpt_adapter_cfg.read_text()).get("r")
            if ckpt_r != p.lora_r:
                log.warning(
                    "Checkpoint lora_r=%s ≠ current lora_r=%s — skipping stale checkpoint",
                    ckpt_r,
                    p.lora_r,
                )
                last_ckpt = None
    try:
        if last_ckpt:
            log.info("Resuming from checkpoint: %s", last_ckpt)
            trainer.train(resume_from_checkpoint=last_ckpt)
        else:
            log.info("No checkpoint found under %s — training from scratch", out_dir)
            trainer.train()
    except Exception:
        import traceback

        err_path = Path(f"{pvc}/logs/rank{rank}-FATAL.txt")
        err_path.parent.mkdir(parents=True, exist_ok=True)
        err_path.write_text(traceback.format_exc())
        log.exception("Training failed — traceback written to %s", err_path)
        raise

    if trainer.is_world_process_zero():
        final = out_dir / "final"
        trainer.save_model(str(final))
        tokenizer.save_pretrained(str(final))
        log.info("Model saved → %s", final)

        # Reopen the HF Trainer's MLflow run (it auto-ends after training).
        from mlflow.tracking import MlflowClient as _MlflowClient

        _mc = _MlflowClient()
        exp = _mc.get_experiment_by_name(p.mlflow_experiment)
        if exp is not None:
            runs = _mc.search_runs(
                experiment_ids=[exp.experiment_id],
                filter_string=f"tags.mlflow.runName = '{training_args.run_name}'",
                max_results=1,
                order_by=["attributes.start_time DESC"],
            )
            if runs:
                _run_id = runs[0].info.run_id
                log.info("Reopening HF Trainer run %s for post-train logging", _run_id)
            else:
                _run_id = None
        else:
            _run_id = None

        with mlflow.start_run(run_id=_run_id, run_name=training_args.run_name):
            # Log final LoRA adapter as a first-class MLflow 3.x LoggedModel
            try:
                model_info = mlflow.transformers.log_model(
                    transformers_model=str(final),
                    name="orpheus-tts-lora",
                    task="text-generation",
                    metadata={
                        "base_model": p.base_model,
                        "adapter_type": "lora",
                        "lora_r": p.lora_r,
                        "dataset": p.hf_dataset,
                    },
                )
                log.info(
                    "LoggedModel → %s (id=%s)",
                    model_info.model_uri,
                    model_info.model_id,
                )
            except Exception as e:
                log.warning(
                    "mlflow.transformers.log_model failed (%s), falling back to artifacts",
                    e,
                )
                meta_files = [
                    "config.json",
                    "tokenizer_config.json",
                    "tokenizer.json",
                    "special_tokens_map.json",
                    "adapter_model.safetensors",
                    "adapter_config.json",
                ]
                for fname in meta_files:
                    meta_path = final / fname
                    if meta_path.exists():
                        mlflow.log_artifact(str(meta_path), artifact_path="model/final")

            mlflow.set_tags({"final_checkpoint_path": str(final), "stage": "final"})
        log.info("Final model tagged in MLflow: %s", final)


print("train_func defined — fully self-contained for inspect.getsource()")

train_func defined — fully self-contained for inspect.getsource()


## Training Client

Authenticate to the cluster. Prefer `NOTEBOOK_USER_TOKEN` from the workbench; otherwise the cell uses the pod service-account token.


In [5]:
# ruff: noqa: F821  # parameters injected by %%yaml parameters
import os
import warnings
from pathlib import Path

import urllib3
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

# OpenShift AI workbenches use a self-signed API cert; suppress the noisy urllib3 warning.
warnings.filterwarnings("ignore", category=urllib3.exceptions.InsecureRequestWarning)

# Prefer workbench env vars; fall back to in-cluster API + SA token.
api_server = os.getenv("OPENSHIFT_API_URL", "https://kubernetes.default.svc")
token = os.getenv("NOTEBOOK_USER_TOKEN")
if not token:
    sa_token = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_token.exists():
        token = sa_token.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench with a service-account token."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.api_key = {"authorization": f"Bearer {token}"}
configuration.verify_ssl = False

api_client = k8s.ApiClient(configuration)
client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=api_client.configuration,
    )
)

## Training Job

Submit a `TransformersTrainer` job with periodic and JIT checkpointing.


In [6]:
# ruff: noqa: F821  # parameters injected by %%yaml parameters
import os

from kubeflow.trainer.options import (
    ContainerOverride,
    Name,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubeflow_trainer_api.models import IoK8sApimachineryPkgApiResourceQuantity

trainer = TransformersTrainer(
    func=train_func,
    func_args=parameters,
    num_nodes=parameters["num_nodes"],
    resources_per_node={
        "nvidia.com/gpu": parameters["gpus_per_node"],
        "cpu": "4",
        "memory": "32Gi",
    },
    env={
        "MLFLOW_TRACKING_URI": os.environ.get(
            "MLFLOW_TRACKING_URI",
            f"http://mlflow.{parameters['namespace']}.svc.cluster.local:5000",
        ),
        "MLFLOW_EXPERIMENT_NAME": parameters["mlflow_experiment"],
        "MLFLOW_TRACKING_INSECURE_TLS": "true",
        "MLFLOW_FLATTEN_PARAMS": "1",
        "HF_MLFLOW_LOG_ARTIFACTS": "0",
    },
    packages_to_install=[
        "peft",
        "snac",
        "soundfile",
        "scipy",
        "librosa",
        "mlflow",
        "jiwer",
        "openai-whisper",
        "pyyaml",
    ],
    output_dir="pvc://shared/orpheus-tts/checkpoints",
    periodic_checkpoint_config=PeriodicCheckpointConfig(
        save_strategy="steps",
        save_steps=parameters["save_steps"],
        save_total_limit=3,
    ),
    enable_jit_checkpoint=True,
    enable_progression_tracking=True,
)

# NCCL needs more than the default 64MB /dev/shm when multiple GPUs share a
# pod. Mount a 1Gi memory-backed volume to avoid "No space left on device".
shm_override = PodTemplateOverrides(
    PodTemplateOverride(
        target_jobs=["node"],
        spec=PodSpecOverride(
            volumes=[
                {
                    "name": "dshm",
                    "emptyDir": {
                        "medium": "Memory",
                        "sizeLimit": IoK8sApimachineryPkgApiResourceQuantity("1Gi"),
                    },
                }
            ],
            containers=[
                ContainerOverride(
                    name="node",
                    volume_mounts=[
                        {
                            "name": "dshm",
                            "mountPath": "/dev/shm",
                        }
                    ],
                )
            ],
        ),
    )
)

runtime = client.backend.get_runtime("torch-distributed")
JOB_NAME = client.train(
    trainer=trainer, runtime=runtime, options=[Name("orpheus-tts"), shm_override]
)
print(f"Job submitted: {JOB_NAME}")

Job submitted: orpheus-tts


## Monitor the training job

Track progress in **OpenShift AI → Training Jobs**, or stream logs below.

> This notebook’s reference job finished **6050/6050** steps, wrote `checkpoints/final`, and reported **Succeeded**. Check `job.status` on your own run.


In [7]:
job = client.get_job(name=JOB_NAME)
print(f"Job: {job.name}")
print(f"Status: {job.status}")

Job: orpheus-tts
Status: Succeeded


In [8]:
for logline in client.get_job_logs(JOB_NAME, follow=True):
    print(logline)

W0811 15:50:55.437000 1 torch/distributed/run.py:852] 
W0811 15:50:55.437000 1 torch/distributed/run.py:852] *****************************************
W0811 15:50:55.437000 1 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0811 15:50:55.437000 1 torch/distributed/run.py:852] *****************************************
[Kubeflow] Initializing checkpoint instrumentation
[Kubeflow] Initializing checkpoint instrumentation
  """Configuration class for applying different quantization configs to modules or parameters based on their fully qualified names (FQNs).
  """Configuration class for applying different quantization configs to modules or parameters based on their fully qualified names (FQNs).
[Kubeflow-node1-rank0] Trainer auto-instrumentation enabled
[Kubeflow] Checkpoint instrumentation enabled
[Ku

## TensorBoard

Browse live scalars in the workbench via the notebook proxy (`NB_PREFIX` + `/proxy/6006/`).


In [9]:
import os

# Route the %tensorboard iframe through the workbench proxy (RHOAI / JupyterHub).
os.environ["TENSORBOARD_PROXY_URL"] = os.environ["NB_PREFIX"] + "/proxy/6006/"
%load_ext tensorboard
%tensorboard --logdir /opt/app-root/src/shared/orpheus-tts/tensorboard

## Resolve LoRA adapter

Use `checkpoints/final` (or the latest `checkpoint-*`) on the shared PVC. Inference loads **base model + LoRA** with PEFT — no merge step.


In [10]:
# ruff: noqa: F821  # parameters injected by %%yaml parameters
import glob
import os

NOTEBOOK_SHARED = "/opt/app-root/src/shared"
hf_cache = f"{NOTEBOOK_SHARED}/orpheus-tts/hf-cache"
ckpt_base = f"{NOTEBOOK_SHARED}/orpheus-tts/checkpoints"
base_model_id = parameters["base_model"]

final_ckpt = os.path.join(ckpt_base, "final")
if os.path.isdir(final_ckpt) and os.path.exists(
    os.path.join(final_ckpt, "adapter_config.json")
):
    adapter_path = final_ckpt
else:
    checkpoints = sorted(glob.glob(os.path.join(ckpt_base, "checkpoint-*")))
    if not checkpoints:
        raise FileNotFoundError(
            f"No checkpoints found at {ckpt_base}. Ensure training completed successfully."
        )
    adapter_path = checkpoints[-1]

print(f"Using LoRA adapter (no merge): {adapter_path}")
print(f"Base model: {base_model_id}")

Using LoRA adapter (no merge): /opt/app-root/src/shared/orpheus-tts/checkpoints/final
Base model: unsloth/orpheus-3b-0.1-pretrained


## Generate Turkish speech

Load **base + LoRA** and synthesize a few Turkish sentences:

1. Tokenize text
2. Generate SNAC audio tokens
3. Decode to 24 kHz waveform


In [11]:
!python3 -m pip install -q snac peft --index-url https://pypi.org/simple

import gc

import IPython.display as ipd
import torch
from peft import PeftModel
from snac import SNAC
from transformers import AutoModelForCausalLM, AutoTokenizer

# Free leftovers from earlier cells (merge attempts, etc.) before loading ~7Gi weights.
for _name in ("ft_model", "base", "model", "snac_model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if (device.type == "cuda" and torch.cuda.is_bf16_supported())
    else torch.float16
)
# Stream weights straight onto GPU — avoids CPU+GPU double residency that OOMs 32Gi workbenches.
device_map = {"": 0} if device.type == "cuda" else "cpu"

LLAMA_VOCAB = 128_256
CODE_OFFSET = LLAMA_VOCAB + 10
SNAC_SR = 24_000
TOK_SOH = LLAMA_VOCAB + 3
TOK_EOH = LLAMA_VOCAB + 4
TOK_SOA = LLAMA_VOCAB + 5
TOK_EOA = LLAMA_VOCAB + 6
TOK_SOS = LLAMA_VOCAB + 1
TOK_EOT = LLAMA_VOCAB + 9

N_CODEBOOK = 4096
N_PER_FRAME = 7

print(
    f"Loading base + LoRA from {adapter_path} (dtype={dtype}, device_map={device_map}) ..."
)
ft_tokenizer = AutoTokenizer.from_pretrained(base_model_id, cache_dir=hf_cache)
ft_tokenizer.pad_token = ft_tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    cache_dir=hf_cache,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    device_map=device_map,
)
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
# PeftModel wraps `base` in-place; do not .to() again (that duplicates).

print("Loading SNAC decoder...")
snac_model = SNAC.from_pretrained(
    "hubertsiuzdak/snac_24khz",
    cache_dir=hf_cache,
).eval()
snac_model = snac_model.to(device)

if device.type == "cuda":
    alloc = torch.cuda.memory_allocated() / (1024**3)
    print(f"GPU allocated after load: {alloc:.2f} GiB")


def build_prompt(text):
    ids = ft_tokenizer.encode(text, add_special_tokens=False) + [TOK_EOT]
    return [TOK_SOH] + ids + [TOK_EOH, TOK_SOA, TOK_SOS]


def snac_decode(token_ids):
    audio_ids = [t for t in token_ids if t >= CODE_OFFSET]
    n = len(audio_ids) // N_PER_FRAME
    if n == 0:
        return None
    audio_ids = audio_ids[: n * N_PER_FRAME]
    l0, l1, l2 = [], [], []
    for f in range(n):
        g = audio_ids[N_PER_FRAME * f : N_PER_FRAME * (f + 1)]
        l0.append((g[0] - CODE_OFFSET) % N_CODEBOOK)
        l1.append((g[1] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[2] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[3] - CODE_OFFSET) % N_CODEBOOK)
        l1.append((g[4] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[5] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[6] - CODE_OFFSET) % N_CODEBOOK)

    def _t(x):
        return torch.tensor(x, dtype=torch.long).unsqueeze(0).to(device)

    with torch.inference_mode():
        wav = snac_model.decode([_t(l0), _t(l1), _t(l2)])
    return wav.squeeze().cpu().float().detach().numpy()


def generate_speech(text, max_new_tokens=900, min_new_tokens=28):
    prompt = build_prompt(text)
    inp = torch.tensor([prompt], dtype=torch.long, device=device)
    with torch.inference_mode():
        out = ft_model.generate(
            inp,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,  # keep low so TOK_EOA can stop early
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            eos_token_id=TOK_EOA,
        )
    new_ids = out[0][len(prompt) :].cpu().tolist()
    del out, inp
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if TOK_EOA in new_ids:
        new_ids = new_ids[: new_ids.index(TOK_EOA)]
    return snac_decode(new_ids)


# Test sentences
test_sentences = [
    ("welcome", "istanbul'a hos geldiniz."),
    ("flight", "sayin yolcularimiz, ucusumuz yaklasik iki saat surecektir."),
    ("farewell", "tesekkur ederiz, iyi yolculuklar dileriz."),
]

for label, text in test_sentences:
    print(f"\n[{label}] {text}")
    wav = generate_speech(text)
    if wav is not None:
        duration = len(wav) / SNAC_SR
        print(f"  Duration: {duration:.2f}s")
        ipd.display(ipd.Audio(wav, rate=SNAC_SR))
    else:
        print("  Failed to generate audio")

Loading base + LoRA from /opt/app-root/src/shared/orpheus-tts/checkpoints/final (dtype=torch.bfloat16, device_map={'': 0}) ...


Loading SNAC decoder...


GPU allocated after load: 6.59 GiB



[welcome] istanbul'a hos geldiniz.
  Duration: 2.22s



[flight] sayin yolcularimiz, ucusumuz yaklasik iki saat surecektir.
  Duration: 5.03s



[farewell] tesekkur ederiz, iyi yolculuklar dileriz.
  Duration: 2.47s


## Cleanup

Delete the TrainJob to free cluster resources. Checkpoints and data remain on the PVC.


In [12]:
client.delete_job(name=JOB_NAME)
print(f"Job {JOB_NAME} deleted")

Job orpheus-tts deleted


## Summary

You used OpenShift AI to fine-tune Orpheus-3B so it can speak Turkish, by training a small LoRA adapter on top of the base model.

This notebook is set up for a full-quality run (all data, stronger LoRA). For a quicker trial, reduce the sample count, number of epochs, or LoRA size in the parameters cell. Training charts and audio samples are tracked in MLflow under the experiment name you set (`mlflow_experiment`).


## Results

This section summarizes how well the Turkish speech model did after the training job in this notebook.

**What we trained:** the full Turkish dataset (~77K clips), for 5 passes over the data, on 4 GPUs, using LoRA adapters (a lightweight way to specialize the model without rewriting all of its weights).

### Did the model learn?

During training we watch **eval loss** — a score for “how surprised” the model is by held-out examples. Lower is better.

| | Score |
| --- | --- |
| Best during training | **~3.96** |
| Near the end of training | ~3.97 |

Loss fell steadily and then leveled off, which is a normal sign that training has largely converged.

### How clear is the speech?

Every so often we ask the model to speak a few fixed Turkish sentences, then use speech recognition (Whisper) to check what it “heard.” We report two everyday error rates:

- **WER** (word error rate) — how many words are wrong  
- **CER** (character error rate) — how many characters are wrong  

Lower is better; **0** would mean a perfect match.

| When we checked | Word errors (WER) | Character errors (CER) |
| --- | --- | --- |
| Mid-training (best clarity) | **~34%** | **~21%** |
| Near the end | ~43% | ~11% |

So the speech became much more intelligible than the English-pretrained starting point. Mid-training checkpoints can sound clearer on these test lines; use a mid-to-late checkpoint if you are picking one to listen to.

**Note:** one end-of-job audio export had extra babble at the end of clips (a generation setting issue). That is fixed in the training code in this notebook; prefer the mid/late checkpoint samples for a fair listen.
